# SHAP Feature Importance

## Set Up

In [ ]:
import sys

sys.path.append("../")
from src.config import BASE_PATH, SEED

In [ ]:
MODEL_LIST = ["xgb", "lgbm", "svc", "lr", "stack", "nn"]
OUTCOME_LIST = ["bleed", "ssi", "pneumo", "reoper", "any", "serious"]
FEAT_ORDER = [
    # Pre-op
    "SEX",
    "RACE_NEW",
    "INOUT",
    "AGE",
    "BMI",
    "DIABETES",
    "SMOKE",
    "FNSTATUS2",
    "HXCOPD",
    "HYPERMED",
    "DISCANCR",
    "WNDINF",
    "WTLOSS",
    "PRALBUM",
    "PRWBC",
    "PRHCT",
    "PRPLATE",
    "ASACLAS",
    "OPERYR",
    ## Inbtra-Op
    "PARTIAL GLOSSECTOMY (HEMIGLOSSECTOMY_SUBTOTAL)",
    "COMPOSITE_EXTENDED GLOSSECTOMY",
    "TOTAL GLOSSECTOMY (COMPLETE TONGUE REMOVAL)",
    "EXCISION OF TONGUE LESIONS (MINOR)",
    "LOCAL_REGIONAL TISSUE FLAPS FOR ORAL CAVITY RECONSTRUCTION",
    "FREE TISSUE TRANSFER (MICROVASCULAR FREE FLAPS) AND COMPLEX FLAP RECONSTRUCTION",
    "SKIN AUTOGRAFTS FOR HEAD AND NECK RECONSTRUCTION",
    "NECK DISSECTION AND LYMPHADENECTOMY PROCEDURES",
    "PERIPHERAL NERVE REPAIR AND NEUROPLASTY",
    "TRACHEOSTOMY PROCEDURES",
    "LARYNGEAL RESECTION AND RECONSTRUCTION PROCEDURES",
    "MALIGNANT NEOPLASM",
]

Ensure we got all bases covered with feat_order

In [ ]:
from src.data_utils import get_data

outcome_dict = get_data(
    "outcome_any", file_dir=BASE_PATH / "data" / "processed_reduced"
)
dummy_df = outcome_dict["X_test"][:5]
all_cols = set()
for col in dummy_df.columns:
    col_split = col.split("_")
    if len(col_split) == 1 or col_split[0] in [
        "PARTIAL GLOSSECTOMY (HEMIGLOSSECTOMY",
        "COMPOSITE",
        "LOCAL",
    ]:
        all_cols.add(col)
    elif col_split[:2] == ["RACE", "NEW"]:
        all_cols.add("RACE_NEW")
    else:
        col_name = col_split[0]
        all_cols.add(col_name)
assert set(FEAT_ORDER) == set(all_cols)

## Run SHAP

In [ ]:
def create_cmd_str(
    model_name,
    feat_order,
    model_path,
    vals_to_explain_path,
    background_vals_path,
    result_path,
):
    feat_list_str = ",".join(feat_order)
    cmd_str = f"export PYTHONPATH={BASE_PATH};\
                uv run python -m src.feat_imp \
                --model_name {model_name}\
                --feat_order '{feat_list_str}'\
                --model_path {model_path}\
                --vals_to_explain_path {vals_to_explain_path}\
                --background_vals_path {background_vals_path}\
                --result_path {result_path}"
    return " ".join(cmd_str.split())

In [ ]:
full_cmd_list = []
for outcome in OUTCOME_LIST:
    swarm_path = BASE_PATH / "swarm_dir" / "shap" "commands" / f"{outcome}.swarm"
    swarm_path.parent.mkdir(exist_ok=True, parents=True)
    if swarm_path.exists():
        swarm_path.unlink()
    outcome_cmd_list = []
    model_base_path = (
        BASE_PATH / "models" / "trained" / outcome
    )  # SHAP on uncalibrated models
    data_base_path = BASE_PATH / "data" / "processed_reduced" / f"outcome_{outcome}"
    for model_abrv in MODEL_LIST:
        background_data_path = data_base_path / "X_train.parquet"
        explanation_data_path = data_base_path / "X_test.parquet"
        result_path = BASE_PATH / "results" / "shap" / outcome / f"{model_abrv}.xlsx"
        if model_abrv == "nn":
            model_path = model_base_path / "nn.pt"
        else:
            model_path = model_base_path / f"{model_abrv}.joblib"
        cmd_str = create_cmd_str(
            model_name=model_abrv,
            feat_order=FEAT_ORDER,
            model_path=model_path,
            vals_to_explain_path=background_data_path,
            background_vals_path=explanation_data_path,
            result_path=result_path,
        )
        outcome_cmd_list.append(cmd_str)
    swarm_path.write_text("\n".join(outcome_cmd_list))
    full_cmd_list += outcome_cmd_list
print(f"Num commands: {len(full_cmd_list)}")

In [ ]:
full_cmd_list

Run local sequentially

In [ ]:
import subprocess

for iteration, cmd in enumerate(full_cmd_list):
    print(f"{iteration +1}/{len(full_cmd_list)}...")
    subprocess.run(cmd, shell=True, check=True)
    break

Run swarms

In [ ]:
# run_swarms(
#     model_list=MODEL_ABRV_LIST,
#     swarm_log_dir=BASE_PATH / "swarm_dir" / "logs",
#     cmd_dir=CMD_DIR,
#     n_jobs=N_CV_JOBS,
#     n_threads=N_THREADS,
#     n_trials=N_TRIALS,
# )